# Perspective bundle adjustment via named-axis autograd

*A paper-style derivation of single-axis bundle adjustment under perspective projection, learned end-to-end via the Python SDK's autograd. The previous notebook ([03_multifocal-tensors](./03_multifocal-tensors.ipynb)) treated the multi-view tensors $F$, $\mathcal{T}$, $\mathcal{Q}$ as algebraic constraints; this notebook works one level lower — recovering camera parameters directly from observed pixel coordinates by minimising the reprojection error.*

The headline upgrade over notebook 03 is the **perspective divide**: image coordinates are $(u, v) = (y^1 / y^3,\; y^2 / y^3)$ where $\mathbf{y} = P\mathbf{X}$. That single division was unavailable until the Phase 6 follow-up extensions PR added `__truediv__` to `DynamicVariable` plus `sin` / `cos` / `sqrt` activations — exactly the surface needed for genuine perspective MVG.

> Educational. Production-grade bundle adjustment uses Levenberg-Marquardt with sparse Jacobians; here we use vanilla SGD with autograd. See [ADR-0001](../../docs/arc42/09-decisions/0001-pivot-to-educational-named-axis-dsl.md) on the production-as-is positioning.

## Prerequisites

- [`00_python-sdk-tour.ipynb`](./00_python-sdk-tour.ipynb), [`01_python-autograd.ipynb`](./01_python-autograd.ipynb), [`03_multifocal-tensors.ipynb`](./03_multifocal-tensors.ipynb).
- Familiarity with the pinhole camera; [Hartley & Zisserman §6](https://www.robots.ox.ac.uk/~vgg/hzbook/) is the textbook reference.

In [ ]:
# Colab / Binder setup — install the tensor SDK on first run.
# On local installs (or Binder where `postBuild` ran), the import works
# without rebuilding. Initial install on Colab takes ~3–5 minutes (the
# nanobind extension compiles on the runner).
try:
    import tensor  # noqa: F401
except ImportError:
    import subprocess as _sp
    _sp.run(
        ["pip", "install", "--quiet",
         "git+https://github.com/uyuutosa/tensor.git"],
        check=True,
    )
    import tensor  # noqa: F401

## 1. Setup

### 1.1 Camera parameterisation

We restrict cameras to **single-axis rotation** around the world's $y$-axis plus a 3-vector translation. The rotation is parameterised by a single scalar $\theta_v \in \mathbb{R}$ via

$$
R(\theta) \;=\;
\begin{pmatrix}
\cos\theta & 0 & -\sin\theta \\
0 & 1 & 0 \\
\sin\theta & 0 & \cos\theta
\end{pmatrix}.
$$

The forward map from a world point $\mathbf{X} = (X^1, X^2, X^3) \in \mathbb{R}^3$ through camera $v = (\theta_v, \mathbf{t}_v)$ to its image coordinates $(u, v) \in \mathbb{R}^2$ is

$$
\begin{aligned}
\mathbf{y} &= R(\theta_v)\,\mathbf{X} + \mathbf{t}_v \quad \in \mathbb{R}^3,\\
(u, v) &= (y^1 / y^3,\;\; y^2 / y^3).
\end{aligned}
$$

Per-camera parameter count: 4 (one angle + three translation components). The 3D structure $\mathbf{X}_n$ is fixed and known throughout — this is the *partial* BA problem (recover cameras only). A *full* BA jointly learns $\mathbf{X}_n$ as well; the modification is one extra `DynamicVariable` per point with `requires_grad=True`.

### 1.2 Loss

Given $N$ observed pixel coordinates $(u^{(v)}_n, v^{(v)}_n)$ across $V$ cameras, the reprojection loss is

$$
\mathcal{L}(\theta_{1:V}, \mathbf{t}_{1:V}) \;=\; \sum_{v=1}^{V}\sum_{n=1}^{N}
\Bigl\Vert
\bigl(u^{(v)}_n - \hat u^{(v)}_n\bigr),\;
\bigl(v^{(v)}_n - \hat v^{(v)}_n\bigr)
\Bigr\Vert_2^2.
$$

Each $(\hat u, \hat v) = (y^1/y^3,\, y^2/y^3)$ is a non-linear function of the camera parameters — quotient of trigonometric polynomials. The chain rule that resolves the gradient through this is exactly what `ag.backward` is for.

In [ ]:
import numpy as np

import tensor
import tensor.autograd as ag

rng = np.random.default_rng(seed=7)

N = 8          # 3D points
V = 3          # cameras

# 3D scene: points in a cube around the origin.
X_np = rng.uniform(-0.5, 0.5, size=(N, 3))

# Ground-truth cameras: 3 viewpoints on a circle of radius 4, each with a
# rotation around y placing the look-axis at the origin.
theta_true = np.array([0.0, 1.5, 3.0])
t_true = np.stack([
    np.array([radius * np.sin(theta), 0.0, radius * np.cos(theta)])
    for theta, radius in zip(theta_true, [4.0, 4.0, 4.0])
])


def rotate_y(theta: float) -> np.ndarray:
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, 0.0, -s], [0.0, 1.0, 0.0], [s, 0.0, c]])


def project_np(theta, t, X_n):
    y = rotate_y(theta) @ X_n + t
    return y[:2] / y[2]


# Ground-truth pixel observations for every (camera, point) pair.
uv_obs = np.zeros((V, N, 2))
for v in range(V):
    for n in range(N):
        uv_obs[v, n] = project_np(theta_true[v], t_true[v], X_np[n])

print(f"true θ:        {theta_true}")
print(f"true t (first view): {t_true[0]}")
print(f"observations shape:  {uv_obs.shape}  (V, N, 2)")

### 3D visualisation of the ground-truth scene

Before training, show the synthetic geometry in 3D: the $N$ world points and the $V$ camera frusta. Interactive `plotly` figures embed inline; if you are reading this on the Jupyter Book site (https://uyuutosa.github.io/tensor/) the figures should rotate, zoom, and tooltip on hover. In a static rendering (PDF / GitHub blob) only a snapshot is visible.

In [ ]:
import plotly.graph_objects as go


def camera_frustum(theta: float, t: np.ndarray, scale: float = 0.5,
                   colour: str = "red") -> go.Mesh3d:
    """Return a plotly Mesh3d showing a camera as a small pyramid.

    World-frame camera centre is C = -R(θ)^T · t (since R · X_world + t
    maps world to camera frame, so the camera centre satisfies
    R · C + t = 0). The look direction is R^T · ẑ.
    """
    R = rotate_y(theta)
    C = -R.T @ t                 # world-frame camera centre
    z_look = R.T @ np.array([0.0, 0.0, 1.0])
    x_axis = R.T @ np.array([1.0, 0.0, 0.0])
    y_axis = R.T @ np.array([0.0, 1.0, 0.0])
    apex = C
    tip = C + scale * z_look
    # 4 corners of the image plane (the pyramid base, perpendicular to z_look)
    half = 0.3 * scale
    corners = np.stack([
        tip + half * (x_axis + y_axis),
        tip + half * (x_axis - y_axis),
        tip + half * (-x_axis - y_axis),
        tip + half * (-x_axis + y_axis),
    ])
    verts = np.vstack([apex.reshape(1, 3), corners])  # (5, 3)
    # 4 side triangles (apex–corner_k–corner_{k+1}) + 2 base triangles
    i = [0, 0, 0, 0, 1, 1]
    j = [1, 2, 3, 4, 2, 3]
    k = [2, 3, 4, 1, 3, 4]
    return go.Mesh3d(
        x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
        i=i, j=j, k=k,
        color=colour, opacity=0.45,
        flatshading=True, showlegend=False, hoverinfo="skip",
    )


# Ground-truth scene: points + cameras.
gt_pts = go.Scatter3d(
    x=X_np[:, 0], y=X_np[:, 1], z=X_np[:, 2],
    mode="markers",
    marker=dict(size=4, color="#1f77b4", opacity=0.9),
    name="3D points (ground truth)",
)
gt_cams = [
    camera_frustum(theta_true[v], t_true[v], scale=0.7, colour="#d62728")
    for v in range(V)
]

fig_setup = go.Figure(data=[gt_pts, *gt_cams])
fig_setup.update_layout(
    title="Synthetic scene — ground-truth points (blue) + cameras (red)",
    scene=dict(
        xaxis_title="x", yaxis_title="y", zaxis_title="z",
        aspectmode="data",
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    height=500,
)
fig_setup.show()

## 2. Autograd-aware projection

Express each camera as 4 scalar `DynamicVariable`s: $\theta_v$ and $(t^1_v, t^2_v, t^3_v)$. For each (camera $v$, point $n$) pair, compute

$$
\begin{aligned}
c &= \cos\theta_v, \quad s = \sin\theta_v, \\
y^1 &= c\,X^1_n - s\,X^3_n + t^1_v, \\
y^2 &= X^2_n + t^2_v, \\
y^3 &= s\,X^1_n + c\,X^3_n + t^3_v, \\
\hat u &= y^1 / y^3, \quad \hat v = y^2 / y^3.
\end{aligned}
$$

The point coordinates $X^i_n$ enter as `DynamicVariable` *constants* (no `requires_grad`); the gradient flows only through the camera parameters.

Each (camera, point) projection therefore touches `ag.cos` / `ag.sin` + 3 multiplications + 4 additions + 2 divisions. With $V \cdot N = 24$ pairs, one forward pass is $\sim 240$ scalar autograd ops + a similar number on the backward — well within Python-loop tractability for the demo scale.

In [ ]:
def _scalar(value: float, *, grad: bool) -> ag.DynamicVariable:
    """Build a rank-0 (scalar) DynamicVariable from a Python float."""
    return ag.DynamicVariable(
        tensor.from_numpy(np.array(value, dtype=np.float64), []),
        requires_grad=grad,
    )


def project_ag(theta, t1, t2, t3, X1, X2, X3):
    """Autograd-aware projection: returns (u_hat, v_hat) as scalar
    DynamicVariables differentiable in θ, t1, t2, t3."""
    c = ag.cos(theta)
    s = ag.sin(theta)
    y1 = c * X1 - s * X3 + t1
    y2 = X2 + t2
    y3 = s * X1 + c * X3 + t3
    return y1 / y3, y2 / y3

## 3. Training loop

Initialise each camera's parameters at perturbed values of the ground truth (so the optimisation has somewhere to go). Train via vanilla SGD on the sum of squared reprojection residuals.

Per epoch: zero the gradients, walk every observation, accumulate $r^2$ into `total_loss`, backward, update.

### Why this is expensive

Each correspondence builds 9 new `DynamicVariable`s on the tape (cos / sin / three `y` components / two image coords / one residual / one squared). With $V\cdot N = 24$ correspondences × 2 image coords each → 48 squared-residual nodes per epoch, plus their backward closures. A `reduce_along_label` collapse would shrink the Python loop but the *constant factor* per autograd-op is the dominant cost; a future Phase 6 follow-up pinning the backward closures' allocator would matter more.

In [ ]:
# Initial guesses: deterministic small perturbation of the ground truth.
# Perspective BA is non-convex; the y¹/y³ quotient produces steep
# gradients when a camera "almost looks down" (y³ → 0), so we keep
# the perturbation small + deterministic so no perturbed camera
# crosses a singular configuration on this scene. With θ_true ∈
# {0, 1.5, 3.0} and the points in [-0.5, 0.5]³, the perturbations
# below keep |y³| ≳ 3 throughout — well clear of the singularity.
theta_init = theta_true + np.array([0.02, -0.03, 0.025])
t_init = t_true + np.array([
    [ 0.05, -0.04,  0.03],
    [-0.03,  0.05, -0.04],
    [ 0.04,  0.03, -0.05],
])

# Wrap each parameter as a scalar DynamicVariable with grad tracking.
thetas = [_scalar(float(theta_init[v]), grad=True) for v in range(V)]
ts = [
    [_scalar(float(t_init[v, k]), grad=True) for k in range(3)]
    for v in range(V)
]

# Pre-wrap the 3D points + observations as scalar DynamicVariables for the
# inner loop. Constants only — no gradient tracking on these.
X_scalars = [
    [_scalar(float(X_np[n, k]), grad=False) for k in range(3)]
    for n in range(N)
]
uv_scalars = [
    [
        (_scalar(float(uv_obs[v, n, 0]), grad=False),
         _scalar(float(uv_obs[v, n, 1]), grad=False))
        for n in range(N)
    ]
    for v in range(V)
]


def reprojection_loss(thetas, ts):
    total = None
    for v in range(V):
        th = thetas[v]
        t1, t2, t3 = ts[v]
        for n in range(N):
            X1, X2, X3 = X_scalars[n]
            u_hat, v_hat = project_ag(th, t1, t2, t3, X1, X2, X3)
            u_obs, v_obs = uv_scalars[v][n]
            du = u_hat - u_obs
            dv = v_hat - v_obs
            r_sq = du * du + dv * dv
            total = r_sq if total is None else total + r_sq
    return total


lr = 0.01    # safe with the small deterministic perturbation above
epochs = 400
history = []
for epoch in range(epochs):
    for th in thetas:
        th.zero_grad()
    for tv in ts:
        for tk in tv:
            tk.zero_grad()
    loss = ag.sum_all(reprojection_loss(thetas, ts))
    ag.backward(loss)
    history.append(float(loss.value))
    # SGD on every parameter.
    thetas = [
        ag.DynamicVariable(ag.sgd_update(th, lr), requires_grad=True)
        for th in thetas
    ]
    ts = [
        [
            ag.DynamicVariable(ag.sgd_update(tk, lr), requires_grad=True)
            for tk in tv
        ]
        for tv in ts
    ]

print(f"epoch   1: loss = {history[0]:.4e}")
print(f"epoch 100: loss = {history[99]:.4e}")
print(f"epoch 400: loss = {history[-1]:.4e}")

## 4. Verification

Compare the recovered camera parameters with the ground truth. The loss should be small ($\sim 10^{-6}$ or better) and each $\theta_v$ and $\mathbf{t}_v$ within $10^{-3}$ of the truth — within SGD's noise floor.

*Note*: bundle adjustment has a **gauge freedom** — the overall scene-camera pose can be rigid-transformed by any element of $SE(3)$ without changing observations. For our 1-DOF cameras the gauge is effectively a global $y$-rotation + a global translation in $(t^1, t^3)$; with $V = 3$ cameras and reasonable initialisation, SGD converges close to the gauge orbit of the ground truth. Real BA pipelines fix gauge explicitly (e.g. clamp the first camera at the origin).

In [ ]:
recovered_theta = np.array([float(th.value.numpy()) for th in thetas])
recovered_t = np.array([
    [float(tk.value.numpy()) for tk in tv]
    for tv in ts
])

print("recovered θ:    ", recovered_theta)
print("ground-truth θ: ", theta_true)
print("|Δθ| max:        ", float(np.max(np.abs(recovered_theta - theta_true))))
print()
print("recovered t[0]:  ", recovered_t[0])
print("ground-truth t[0]:", t_true[0])
print("|Δt| max:        ", float(np.max(np.abs(recovered_t - t_true))))

# Sanity floor — the loss should drop ≥ 2 orders of magnitude from its
# initial value. We do not assert absolute parameter match because of the
# SE(3) gauge ambiguity discussed above; loss-reduction is the right
# success criterion for an unfixed-gauge BA.
initial_loss = history[0]
final_loss = history[-1]
ratio = initial_loss / max(final_loss, 1e-30)
print(f"\ninitial loss {initial_loss:.4e} → final loss {final_loss:.4e}")
print(f"reduction factor: {ratio:.1f}×")
assert ratio >= 100.0, (
    f"Reprojection loss did not drop ≥ 100×: ratio = {ratio:.2f}× "
    f"(initial {initial_loss:.4e}, final {final_loss:.4e})"
)

### Loss curve + recovered geometry

Plot the per-epoch reprojection-loss decay (semilog y) and the recovered camera frusta overlaid on the ground truth. Recovered cameras (green) should coincide with the ground-truth ones (red) up to the SE(3) gauge freedom.

In [ ]:
import matplotlib.pyplot as plt

# Loss curve.
fig_loss, ax_loss = plt.subplots(figsize=(6, 3))
ax_loss.semilogy(history)
ax_loss.set_xlabel("epoch")
ax_loss.set_ylabel("reprojection loss")
ax_loss.set_title(f"reprojection loss (initial {history[0]:.2e} → final {history[-1]:.2e})")
ax_loss.grid(True, which="both", linestyle=":", alpha=0.5)
plt.show()

# 3D overlay of ground-truth vs recovered cameras + the points.
recovered_cams = [
    camera_frustum(recovered_theta[v], recovered_t[v], scale=0.7, colour="#2ca02c")
    for v in range(V)
]
fig_overlay = go.Figure(data=[
    gt_pts,
    *gt_cams,           # red — ground truth
    *recovered_cams,    # green — recovered
])
fig_overlay.update_layout(
    title="Overlay — ground-truth (red) vs recovered (green) cameras",
    scene=dict(
        xaxis_title="x", yaxis_title="y", zaxis_title="z",
        aspectmode="data",
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    height=500,
)
fig_overlay.show()

## 5. What this notebook demonstrates

1. **Perspective projection inside the autograd graph**. The expression $y^1 / y^3$ is a `DynamicVariable.__truediv__` call; its backward closure carries the quotient-rule gradient $\partial(y^1/y^3)/\partial y^1 = 1/y^3$ and $\partial(y^1/y^3)/\partial y^3 = -y^1/(y^3)^2$, both with broadcast-aware unbroadcast. This was the missing surface between [notebook 03](./03_multifocal-tensors.ipynb) (multilinear-only) and real geometric vision.
2. **Trigonometric rotation parameters as differentiable scalars**. `ag.cos(theta)` and `ag.sin(theta)` produce rank-0 `DynamicVariable`s that compose with the rest of the graph; the chain rule resolves through them to the underlying $\theta_v$ scalars.
3. **Named-axis discipline in the small case**. Even at the scalar-per-parameter scale of this demo, every $X^i$, $t^k$, $y^j$ is a labelled `DynamicVariable` — the Python identifier and the mathematical symbol coincide. A larger BA (more cameras, joint structure-and-motion, multi-axis rotations via Rodrigues) would graduate to rank-1 and rank-2 named tensors but the same vocabulary.

### What this notebook does *not* demonstrate

- **Full Rodrigues rotation**. The 1-axis $R_y(\theta)$ keeps the demo small but real BA uses the 3-DOF axis-angle parameterisation
  $$R(\boldsymbol{\omega}) = I + \frac{\sin\|\boldsymbol{\omega}\|}{\|\boldsymbol{\omega}\|}[\boldsymbol{\omega}]_\times + \frac{1 - \cos\|\boldsymbol{\omega}\|}{\|\boldsymbol{\omega}\|^2}[\boldsymbol{\omega}]_\times^2,$$
  which needs all of `ag.sin` + `ag.cos` + `ag.sqrt` (for $\|\boldsymbol{\omega}\|$) + `__truediv__` — every extension this Bundle B PR adds. Building it on top of this notebook's machinery is *mechanical*: replace the 3×3 $R_y$ block with the 9 Rodrigues entries computed elementwise.
- **Robust losses**. A real photogrammetry pipeline uses Huber / Cauchy / Geman-McClure to suppress outliers. The squared loss above breaks on a single bad correspondence.
- **Sparse Jacobians**. The autograd here is dense; production BA exploits the block-diagonal structure of the Jacobian. For pedagogical exposition that structural sparsity is invisible; for million-point reconstruction it is everything.
- **Levenberg-Marquardt**. Vanilla SGD on a non-convex landscape is unreliable; LM with trust-region updates is the genre default. The autograd machinery above generalises to LM (the Gauss-Newton step needs Jacobian-vector products, which `ag.backward` already provides).

## 6. References

- **R. Hartley & A. Zisserman**, *Multiple View Geometry in Computer Vision*, 2nd ed., CUP 2004. Chapter 6 (Camera Models), Chapter 7 (Computation of the Camera Matrix), Chapter 18 (N-View Computational Methods §18.1 — Bundle Adjustment).
- **B. Triggs, P. McLauchlan, R. Hartley, A. Fitzgibbon**, *Bundle Adjustment — A Modern Synthesis*, Workshop on Vision Algorithms, ICCV 1999. The canonical BA paper.
- **C. Olsson, A. Eriksson, F. Kahl**, *Improved Initialization for Bundle Adjustment*, ICCV 2007. The non-convexity problem this notebook's perturbation-of-truth initialisation conveniently dodges.
- **K. Madsen, H. B. Nielsen, O. Tingleff**, *Methods for Non-Linear Least Squares Problems*, 2004 lecture notes (DTU). The Levenberg-Marquardt + trust-region reference for the genre default.

## 7. Phase 6 follow-ups surfaced by this notebook

- Bind `tensor::core::reduce_along_labels` *only this notebook still uses scalar-per-parameter*; once Phase 6 binds it the inner loop collapses to a single batched reprojection-vector + a single `reduce_along_label` over the (camera, point) axes.
- Levenberg-Marquardt helper (Phase 6 stretch goal). The Gauss-Newton step is $\Delta = -(J^T J + \lambda I)^{-1} J^T r$; `ag.backward` produces $J^T r$ already, only the $J^T J$ inverse is new.
- A 3-DOF Rodrigues helper as a reusable Python function would let downstream notebooks reach for full perspective + 6-DOF cameras without rebuilding $R(\boldsymbol{\omega})$ each time.